# Kraken Spot API Interaction Example

Interacts with the Kraken Spot REST API via the `SpotConnector`.

## Setup

In [ ]:
import os

from dotenv import load_dotenv

from kraken_kit import SpotConnector

load_dotenv()

spot = SpotConnector(
    api_key=os.environ["KRAKEN_API_KEY"],
    api_secret=os.environ["KRAKEN_API_SECRET"],
)

## Account Balance

In [ ]:
spot.get_balance()

## Available Trading Pairs

In [ ]:
pairs = spot.get_asset_pairs()
print(f"{len(pairs)} pairs available")
for name in list(pairs.keys())[:10]:
    print(f"  {name}")

## Symbol Info and Ticker

In [ ]:
symbol = "XBTUSDC" 
ticker = spot.get_ticker(symbol)
ticker

## Place a Market Order

In [ ]:
side = "buy"
qty = 5.0 / ticker["mid"]     # $10 worth of BTC — truncated internally by the connector to meet kraken requirements
validate = False               # dry run — order is validated but not submitted

spot.place_order(symbol, side, qty, validate=validate)

## Place a Limit Order

In [ ]:
side = "buy"
qty = 5.0 / ticker["mid"]
price = 50000 
validate = False

spot.place_order(symbol, side, qty, price=price, validate=validate)

In [ ]:
# Time-in-force options for limit orders (default is GTC)
# spot.place_order(symbol, side, qty, price=price, time_in_force="GTC")   # Good-til-cancelled — stays open until filled or cancelled
# spot.place_order(symbol, side, qty, price=price, time_in_force="IOC")   # Immediate-or-cancel — fills what it can, cancels the rest
# spot.place_order(symbol, side, qty, price=price, time_in_force="GTD", expiretm="+3600")   # Good-til-date — expires at a set time "expiretm" set in seconds

## Conditional Close — Stop-Loss

In [ ]:
side = "buy"
qty = 5.0 / ticker["mid"]
price = 50000
close_type = "stop-loss"
# close_price = 45000  # absolute price
close_price = "-5%"  # or relative: 5% below fill price
# close_price = "#500"  # or offset: $500 below fill price
validate = False

spot.place_order_with_close(
    symbol=symbol,
    side=side,
    volume=qty,
    close_type=close_type,
    close_price=close_price,
    price=price,
    validate=validate,
)

## Conditional Close - Take Profit

In [ ]:
side = "buy"
qty = 5.0 / ticker["mid"]
# price = 50000
close_type = "take-profit"
# close_price = "+5%"  # relative: 5% above fill price
# close_price = 55000.0  # or absolute price
close_price = "#1000"  # or offset: $1000 above fill price
validate = False

spot.place_order_with_close(
    symbol=symbol,
    side=side,
    volume=qty,
    close_type=close_type,
    close_price=close_price,
    # price=price,              
    validate=validate,
)

## Conditional Close — Limit Variants

In [ ]:
side = "buy"
qty = 5.0 / ticker["mid"]
price = 70000

# Stop-loss-limit: places a limit order at close_limit_price, stop loss triggers at close_price
close_type = "stop-loss-limit"
close_price = 65000
close_limit_price = 65500

# Take-profit-limit: places a limit order at close_limit_price, take profit triggers at close_price
# close_type = "take-profit-limit"
# close_price = 85000
# close_limit_price = 84500

validate = False

spot.place_order_with_close(
    symbol=symbol,
    side=side,
    volume=qty,
    close_type=close_type,
    close_price=close_price,
    price=price,
    close_limit_price=close_limit_price,
    validate=validate,
)

## Standalone Conditional Orders

In [ ]:
# spot.place_order(symbol, "sell", qty, price=45000, ordertype="stop-loss", validate=False)

# spot.place_order(symbol, "sell", qty, price=45000, ordertype="stop-loss-limit", trigger_price=44500, validate=False)

# spot.place_order(symbol, "sell", qty, price=90000, ordertype="take-profit", validate=False)

# spot.place_order(symbol, "sell", qty, price=90000, ordertype="take-profit-limit", trigger_price=89500, validate=False)

## Standalone Conditional Orders - Trailing Stop

In [ ]:
qty = 0.00006
# spot.place_order(symbol, "sell", qty, price="+5%", ordertype="trailing-stop", validate=False)

# spot.place_order(symbol, "sell", qty, price="+5%", ordertype="trailing-stop-limit", trigger_price="-1%", validate=False)

## Batch Orders

In [ ]:
qty = 5 / ticker["mid"]
validate = False

orders = [
    {"symbol": symbol, "ordertype": "limit", "type": "buy", "price": "65000", "volume": str(qty)},
    {"symbol": symbol, "ordertype": "limit", "type": "buy", "price": "64000", "volume": str(qty)},
]

spot.place_order_batch(orders, validate=validate)

## Cancel, Edit, and Amend

Place a far-from-market limit order, then demonstrate cancel/edit/amend.

**Edit** = cancel-and-replace (new txid, loses queue priority).  
**Amend** = in-place modification (same txid, preserves queue priority).  
Neither works on orders with conditional close.

In [ ]:
# Place a limit order far from market — it won't fill
side = "buy"
qty = 5.0 / ticker["mid"]
price = 50000

txid = spot.place_order(symbol, side, qty, price=price)
print(f"Order placed: {txid}")

In [ ]:
# Edit: cancel-and-replace with a new price → returns a new txid
new_txid = spot.edit_order(txid, symbol="XBTUSDC", limit_price=1100)
print(f"Edited → new txid: {new_txid}")
txid = new_txid

In [ ]:
# Amend: modify in-place, keep queue priority
amend_id = spot.amend_order(txid, limit_price="1200")
print(f"Amended: {amend_id}")

In [ ]:
spot.cancel_order(txid)

## Cancel All & Dead Man's Switch

The dead man's switch cancels all orders after a timeout.
Call it periodically (every 15–30 s) with a 60 s timeout to keep orders alive.
Pass `0` to deactivate.

In [ ]:
# Activate: cancel everything if no heartbeat within 60s
spot.cancel_all_after(60)

In [ ]:
# Deactivate
spot.cancel_all_after(0)